# WESAD HRV Stress Detection - Specificity & Uncertainty (v3)

Adds, on top of the main study and v2:
1. **Three-class LOSO** (baseline vs stress vs amusement) with a confusion matrix, to see
   *where* errors fall - in particular whether stress is confused with amusement.
2. **Stress-vs-amusement discrimination**: both are high-arousal states, so this is a direct
   test of whether the model detects *stress* specifically rather than generic arousal.
3. **Per-subject F1 + bootstrap 95% confidence intervals** on the headline baseline-vs-stress
   task, so the reported performance carries an honest uncertainty estimate.

Run: open in Colab -> Runtime -> Run all. ~15-25 min (PPG peak detection dominates).


In [ ]:
# 1. Dependency (pin only neurokit2; others are Colab defaults)
!pip -q install neurokit2==0.2.10

In [ ]:
# 2. Download + unzip WESAD (~2 GB, in Colab)
import os, zipfile
DATA_DIR="/content/wesad"; ZIP_PATH="/content/WESAD.zip"
WESAD_URL="https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATA_DIR,"WESAD","S2","S2.pkl")):
    if not os.path.exists(ZIP_PATH):
        print("Downloading WESAD (~2 GB)..."); import subprocess
        subprocess.run(["wget","-q","-O",ZIP_PATH,WESAD_URL])
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)
print("Subjects:", sorted(os.listdir(os.path.join(DATA_DIR,"WESAD"))))

In [ ]:
# 3. Imports + configuration
import pickle, warnings, sys
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import neurokit2 as nk, sklearn, scipy
from scipy import signal as spsig, interpolate as spinterp
from scipy.integrate import trapezoid
from tqdm import tqdm

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__, "| neurokit2", nk.__version__)

WESAD_ROOT=os.path.join(DATA_DIR,"WESAD")
SUBJECTS=[f"S{i}" for i in [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]]
BVP_FS=64; LABEL_FS=700
WINDOW_SEC=60; STRIDE_SEC=30
# 1=baseline, 2=stress, 3=amusement  ->  0,1,2
LABEL_MAP={1:0, 2:1, 3:2}
CLASS_NAMES={0:"baseline",1:"stress",2:"amusement"}
PURITY=0.90; MIN_BEATS=12
OUT="/content/outputs_v3"; os.makedirs(OUT, exist_ok=True)
PPG_FEATS=["meanHR","SDNN","RMSSD","pNN50","LF","HF","LFHF"]

In [ ]:
# 4. HRV helpers (time + frequency), identical pipeline to the main study
def freq_domain(rri):
    t=np.cumsum(rri)/1000.0; t=t-t[0]
    if t[-1]<20 or len(rri)<8: return np.nan,np.nan,np.nan
    ti=np.arange(0,t[-1],0.25)
    x=spinterp.interp1d(t,rri,kind="cubic",fill_value="extrapolate")(ti); x=x-x.mean()
    f,p=spsig.welch(x,fs=4.0,nperseg=min(len(x),256))
    lf=trapezoid(p[(f>=0.04)&(f<0.15)], f[(f>=0.04)&(f<0.15)])
    hf=trapezoid(p[(f>=0.15)&(f<0.40)], f[(f>=0.15)&(f<0.40)])
    return lf,hf,(lf/hf if hf>0 else np.nan)

def hrv_from_rri(rri):
    rri=rri[(rri>=300)&(rri<=2000)]
    if len(rri)<MIN_BEATS-1: return None
    d=np.diff(rri); lf,hf,lfhf=freq_domain(rri)
    return dict(meanHR=60000.0/np.mean(rri), SDNN=np.std(rri,ddof=1),
                RMSSD=np.sqrt(np.mean(d**2)), pNN50=np.mean(np.abs(d)>50)*100,
                LF=lf, HF=hf, LFHF=lfhf)

def ppg_rri(bvp):
    try:
        c=nk.ppg_clean(bvp, sampling_rate=BVP_FS)
        pk=nk.ppg_findpeaks(c, sampling_rate=BVP_FS)["PPG_Peaks"]
    except Exception: return None
    if len(pk)<MIN_BEATS: return None
    return np.diff(pk)/BVP_FS*1000.0

In [ ]:
# 5. Window every subject for baseline / stress / amusement
rows=[]
for subj in tqdm(SUBJECTS, desc="subjects"):
    with open(os.path.join(WESAD_ROOT,subj,f"{subj}.pkl"),"rb") as fh:
        data=pickle.load(fh, encoding="latin1")
    bvp=np.asarray(data["signal"]["wrist"]["BVP"]).flatten()
    label=np.asarray(data["label"]).flatten()
    wlen, stride = WINDOW_SEC*BVP_FS, STRIDE_SEC*BVP_FS
    for i in range(0, len(bvp)-wlen, stride):
        t0,t1=i/BVP_FS,(i+wlen)/BVP_FS
        lab=label[int(t0*LABEL_FS):int(t1*LABEL_FS)]
        if len(lab)==0: continue
        v,c=np.unique(lab,return_counts=True); dom=int(v[np.argmax(c)]); pur=c.max()/c.sum()
        if dom not in LABEL_MAP or pur<PURITY: continue
        prri=ppg_rri(bvp[i:i+wlen])
        feats=hrv_from_rri(prri) if prri is not None else None
        if feats is None: continue
        feats.update(label=LABEL_MAP[dom], subject=subj)
        rows.append(feats)

df=pd.DataFrame(rows).dropna(subset=PPG_FEATS).reset_index(drop=True)
df.to_csv(f"{OUT}/features_v3.csv",index=False)
print("Kept windows:", df.shape)
print("Class balance:", {CLASS_NAMES[k]:int(v) for k,v in df["label"].value_counts().items()})

In [ ]:
# 6. Shared LOSO helpers
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, recall_score

LR=lambda:LogisticRegression(max_iter=1000)
RF=lambda:RandomForestClassifier(n_estimators=300,random_state=42)

def loso_predict(data, cols, mk):
    """Return per-fold (subject) true/pred arrays across all LOSO folds."""
    y_true=[]; y_pred=[]; subj_f1=[]; subjects=[]
    for ts in sorted(data.subject.unique()):
        tr=data[data.subject!=ts]; te=data[data.subject==ts]
        if te.empty or tr.label.nunique()<2 or te.label.nunique()<1: continue
        sc=StandardScaler().fit(tr[cols]); m=mk().fit(sc.transform(tr[cols]),tr.label)
        pred=m.predict(sc.transform(te[cols]))
        y_true.append(te.label.values); y_pred.append(pred)
        subj_f1.append(f1_score(te.label,pred,average="macro",zero_division=0)); subjects.append(ts)
    return y_true,y_pred,np.array(subj_f1),subjects

In [ ]:
# 7. THREE-CLASS LOSO (baseline vs stress vs amusement) + confusion matrix
yt,yp,sf1,subs=loso_predict(df, PPG_FEATS, RF)
YT=np.concatenate(yt); YP=np.concatenate(yp)
labels3=[0,1,2]
cm=confusion_matrix(YT,YP,labels=labels3)
print("3-class RF  macro-F1 = %.3f  (per-subject mean %.3f +/- %.3f)"%(
      f1_score(YT,YP,average="macro"), sf1.mean(), sf1.std()))
print("3-class RF  accuracy = %.3f"%accuracy_score(YT,YP))
print("\nConfusion matrix (rows=true, cols=pred): baseline / stress / amusement")
cmdf=pd.DataFrame(cm,index=[CLASS_NAMES[i] for i in labels3],columns=[CLASS_NAMES[i] for i in labels3])
print(cmdf.to_string())
cmnorm=cm/cm.sum(axis=1,keepdims=True)
print("\nRow-normalised (recall per class):")
print(pd.DataFrame(cmnorm.round(3),index=cmdf.index,columns=cmdf.columns).to_string())
# also majority baseline for 3-class
maj=int(df.label.mode()[0]); print("\n3-class majority-baseline macro-F1 = %.3f"%
      f1_score(df.label, np.full(len(df),maj), average="macro", zero_division=0))
cmdf.to_csv(f"{OUT}/confusion_3class.csv")

In [ ]:
# 8. STRESS vs AMUSEMENT (specificity: both high-arousal) LOSO
sa=df[df.label.isin([1,2])].copy()   # 1=stress, 2=amusement
# relabel to 0/1 for clean binary metrics: stress=1, amusement=0
sa["y"]=(sa.label==1).astype(int)
def loso_bin(data,cols,mk,ycol="y"):
    f1s=[];accs=[];rec=[]
    for ts in sorted(data.subject.unique()):
        tr=data[data.subject!=ts]; te=data[data.subject==ts]
        if te.empty or tr[ycol].nunique()<2 or te[ycol].nunique()<1: continue
        sc=StandardScaler().fit(tr[cols]); m=mk().fit(sc.transform(tr[cols]),tr[ycol])
        pred=m.predict(sc.transform(te[cols]))
        f1s.append(f1_score(te[ycol],pred,average="macro",zero_division=0))
        accs.append(accuracy_score(te[ycol],pred))
        rec.append(recall_score(te[ycol],pred,pos_label=1,zero_division=0))
    return np.array(f1s),np.array(accs),np.array(rec)
print("Stress-vs-amusement window counts:", {CLASS_NAMES[k]:int(v) for k,v in sa.label.value_counts().items()})
for nm,mk in [("Logistic Regression",LR),("Random Forest",RF)]:
    f1s,accs,rec=loso_bin(sa,PPG_FEATS,mk)
    print("%s: macro-F1=%.3f +/- %.3f | acc=%.3f | stress-recall=%.3f | folds=%d"%(
          nm,f1s.mean(),f1s.std(),accs.mean(),rec.mean(),len(f1s)))
# majority baseline
majsa=int(sa.y.mode()[0])
print("Majority baseline macro-F1=%.3f"%f1_score(sa.y,np.full(len(sa),majsa),average="macro",zero_division=0))

In [ ]:
# 9. PER-SUBJECT F1 + BOOTSTRAP 95% CI on headline baseline-vs-stress (RF)
bs=df[df.label.isin([0,1])].copy()   # baseline vs stress
_,_,subj_f1,subj_ids=loso_predict(bs, PPG_FEATS, RF)
order=np.argsort(subj_f1)
print("Per-subject macro-F1 (baseline vs stress, RF):")
for idx in order:
    print("  %-4s  %.3f"%(subj_ids[idx], subj_f1[idx]))
rng=np.random.default_rng(42); B=5000
boot=np.array([rng.choice(subj_f1,size=len(subj_f1),replace=True).mean() for _ in range(B)])
lo,hi=np.percentile(boot,[2.5,97.5])
print("\nMean per-subject macro-F1 = %.3f"%subj_f1.mean())
print("Bootstrap 95%% CI (over %d subjects, B=%d) = [%.3f, %.3f]"%(len(subj_f1),B,lo,hi))
print("Min / median / max subject F1 = %.3f / %.3f / %.3f"%(subj_f1.min(),np.median(subj_f1),subj_f1.max()))
pd.DataFrame({"subject":subj_ids,"macro_f1":subj_f1}).to_csv(f"{OUT}/per_subject_f1.csv",index=False)

In [ ]:
# 10. Confusion-matrix figure (3-class) for the paper
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(4.6,4.0))
im=ax.imshow(cmnorm,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels([CLASS_NAMES[i] for i in labels3]); ax.set_yticklabels([CLASS_NAMES[i] for i in labels3])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("3-class LOSO confusion (row-normalised)")
for i in range(3):
    for j in range(3):
        ax.text(j,i,f"{cmnorm[i,j]:.2f}",ha="center",va="center",
                color="white" if cmnorm[i,j]>0.5 else "black",fontsize=11)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_confusion3.png",dpi=200); plt.show()
print("saved:", sorted(os.listdir(OUT)))

In [ ]:
# 11. Summary block to paste into the paper
L=["## v3 results (specificity + uncertainty)",""]
L.append("### 3-class LOSO (baseline/stress/amusement), RF")
L.append("- macro-F1 = %.3f, accuracy = %.3f, majority-baseline macro-F1 = %.3f"%(
         f1_score(YT,YP,average="macro"), accuracy_score(YT,YP),
         f1_score(df.label, np.full(len(df),int(df.label.mode()[0])), average="macro", zero_division=0)))
L.append("- per-class recall: "+", ".join("%s=%.3f"%(CLASS_NAMES[i],cmnorm[i,i]) for i in labels3))
L.append("- stress->amusement confusion = %.3f ; amusement->stress confusion = %.3f"%(cmnorm[1,2],cmnorm[2,1]))
L+=["","### Stress vs amusement (both high-arousal), LOSO"]
for nm,mk in [("LR",LR),("RF",RF)]:
    f1s,accs,rec=loso_bin(sa,PPG_FEATS,mk)
    L.append("- %s: macro-F1=%.3f+/-%.3f, acc=%.3f, stress-recall=%.3f"%(nm,f1s.mean(),f1s.std(),accs.mean(),rec.mean()))
L.append("- majority baseline macro-F1=%.3f"%f1_score(sa.y,np.full(len(sa),int(sa.y.mode()[0])),average="macro",zero_division=0))
L+=["","### Baseline vs stress (headline), per-subject bootstrap"]
L.append("- mean per-subject macro-F1 = %.3f, 95%% CI [%.3f, %.3f]"%(subj_f1.mean(),lo,hi))
L.append("- min/median/max subject F1 = %.3f/%.3f/%.3f"%(subj_f1.min(),np.median(subj_f1),subj_f1.max()))
L.append("- windows: "+str({CLASS_NAMES[k]:int(v) for k,v in df.label.value_counts().items()}))
summary="\n".join(L); open(f"{OUT}/summary_v3.md","w").write(summary); print(summary)

## Next steps
- Download `outputs_v3/` (summary_v3.md, confusion_3class.csv, per_subject_f1.csv, fig_confusion3.png).
- Fold the specificity + CI numbers into the paper.
